# 3. Per-cell tangent basis

Builds a tangent basis for every cell of the LA mesh from a smooth parameterization of the whole mesh. The basis gives a consistent local 2D frame on the surface, which is what allows vector fields (e.g. fibers) to be expressed and transferred between meshes.

Output: `basis_vecs.npy` in the patient directory.

In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv

from uacpy import basis
from uacpy.common import mesh_utils
from uacpy.output import logging, visualization

## Load the mesh

Reads the tagged mesh as triangular `PolyData`. `simplices` is the `(n_cells, 3)` array of vertex indices, extracted from pyvista's flattened face format (`[3, i, j, k, ...]`).

In [ ]:
patient_id = "07"
data_dir = Path(f"../data/patient_{patient_id}")
input_mesh_file = data_dir / "mesh_with_fibers_tags.vtu"
anatomical_tags = {"MV": 0, "LAA": 1, "LIPV": 2, "LSPV": 3, "RIPV": 4, "RSPV": 5}

mesh = pv.read(input_mesh_file)
mesh = mesh_utils.convert_unstructured_to_polydata_mesh(mesh)
simplices = np.array(mesh.faces.reshape(-1, 4)[:, 1:4])

## Construct the basis

`construct_cellwise_basis` builds one global, smooth 2D parameterization of the whole mesh and derives each cell's tangent basis from it:

1. **Close the PV holes.** The raw LA mesh has several boundary loops (the MV plus one per PV opening), but mean value coordinates need a topological disk with exactly one boundary. The PV openings are therefore temporarily filled in.
2. **Map to a disk with the MV boundary.** The MV loop is now the disk's only boundary and is mapped to a circle of radius `exterior_boundary_radius` (in parameterization units). Mean value coordinates (Floater) then place every interior vertex inside the circle, which gives a smooth, fold-over-free 2D embedding of the whole mesh.
3. **Open the holes again.** The coordinates of the added fill vertices are discarded and only the original mesh's triangles are kept, so the PV openings are back.
4. **Derive the basis.** For each original triangle, the Jacobian of the 3D→2D map is computed and its two columns are orthonormalized into a tangent-plane basis.

Returns:

- `basis_vecs`: shape `(n_cells, 3, 2)`, each cell's orthonormal tangent basis (the last axis indexes the two directions),
- `coordinates_2d`: shape `(n_vertices, 2)`, the underlying global parameterization,
- `cellwise_jacobians`: shape `(n_cells, 3, 2)`, the raw Jacobians before orthonormalization.

Because all cells' bases come from this single smooth parameterization, neighboring cells get consistent, slowly varying frames. The basis vectors are saved for later use.

In [ ]:
logger = logging.BaseLogger(logging.LoggerSettings(logfile_path=data_dir / "basis.log"))
basis_vecs, coordinates_2d, cellwise_jacobians = basis.construct_cellwise_basis(
    mesh, anatomical_tags["MV"], exterior_boundary_radius=10, logger=logger
)
np.save(data_dir / "basis_vecs.npy", basis_vecs)

## Visualize the basis

The first basis direction is drawn in blue and the second in red. A good basis varies smoothly over the surface, with no abrupt flips between neighboring cells.

In [ ]:
visualization.plot_mesh_with_vector_field(
    mesh, basis_vecs[:, :, 0], scaling_factor=1, vector_color="blue"
)
visualization.plot_mesh_with_vector_field(
    mesh, basis_vecs[:, :, 1], scaling_factor=1, vector_color="red"
)